In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# BlindDetection-V1 image-only callback
Prepared only. This notebook is fixed N=4 engineering evidence with science denominator zero; it must consume an already frozen N_dev=256 threshold.

In [ ]:
import os
from pathlib import Path
import re
import subprocess
import torch

EXECUTION_EXACT = os.environ['CEGWM_BLIND_DETECTION_EXACT']
if re.fullmatch(r'[0-9a-f]{40}', EXECUTION_EXACT) is None:
    raise ValueError('bind an authorized exact commit')
if not torch.cuda.is_available():
    raise RuntimeError('GPU required; callback not executed')
N_CALLBACK = 4
CHECKOUT = Path('/content/CEG-WM-BlindDetection-V1')
if CHECKOUT.exists():
    raise FileExistsError('detached callback checkout must be create-only')
subprocess.run(['git', 'clone', 'https://github.com/RICHAAARC/CEG-WM.git', str(CHECKOUT)], check=True)
subprocess.run(['git', '-C', str(CHECKOUT), 'checkout', '--detach', EXECUTION_EXACT], check=True)
if subprocess.check_output(['git', '-C', str(CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip() != EXECUTION_EXACT:
    raise RuntimeError('detached checkout identity differs')


In [ ]:
CALLBACK_ROOT = Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V1/callback-input')
MANIFEST = CALLBACK_ROOT / 'manifest-n4.json'
KEY_FILE = CALLBACK_ROOT / 'detection-key.bin'
THRESHOLD = Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V1/calibration') / EXECUTION_EXACT / 'blind_detection_v1_thresholds.json'
OUTPUT = Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V1/callback') / EXECUTION_EXACT / 'result.json'
RUNTIME_FACTORY = os.environ['CEGWM_BLIND_RUNTIME_FACTORY']
if not MANIFEST.is_file() or not KEY_FILE.is_file() or not THRESHOLD.is_file():
    raise FileNotFoundError('frozen N=4 manifest, key file, or threshold asset is absent')
manifest = __import__('json').loads(MANIFEST.read_text(encoding='utf-8'))
if manifest.get('denominator') != N_CALLBACK or len(manifest.get('cases', [])) != N_CALLBACK:
    raise ValueError('callback denominator must remain exactly four')
coverage = {case.get('coverage') for case in manifest['cases']}
required = {'direct_positive', 'geometry_recovered_positive', 'unwatermarked_geometry_negative'}
if not required.issubset(coverage):
    raise ValueError('callback coverage differs')
if OUTPUT.exists():
    raise FileExistsError('callback result is create-only')
if 'RUNNER_CALLS' not in globals():
    RUNNER_CALLS = 0


In [ ]:
assert RUNNER_CALLS == 0
command = [
    'python', str(CHECKOUT / 'experiments/run_blind_detection_v1.py'), 'callback',
    '--manifest', str(MANIFEST), '--key-file', str(KEY_FILE),
    '--threshold', str(THRESHOLD), '--runtime-factory', RUNTIME_FACTORY,
    '--output', str(OUTPUT),
]
RUNNER_CALLS += 1
completed = subprocess.run(command, check=False)
if RUNNER_CALLS != 1 or not OUTPUT.is_file():
    raise RuntimeError('OPERATIONAL_BLOCKED: callback did not retain a create-only result')
result = __import__('json').loads(OUTPUT.read_text(encoding='ascii'))
if result.get('status') == 'METHOD_FAILED':
    raise RuntimeError('METHOD_FAILED: frozen tau and N=4 route gates did not pass')
if result.get('status') == 'OPERATIONAL_BLOCKED':
    raise RuntimeError('OPERATIONAL_BLOCKED: inspect retained callback records')
if completed.returncode != 0 or result.get('status') != 'CALLBACK_N4_PASSED':
    raise RuntimeError('callback status/exit contract differs')
